# SI section 1: Which vertex to monitor
Each cell prints one sentence of the section and every number the sentence quotes.
The column SI is the quoted value and the column computed is the value at the precision of the SI.
Data: `python -m fixloc.make.si.monitoring` for the graphs with $N=7$, $9$, $10$, and $13$ vertices,
`fixloc.large` on a cluster for the simulated networks, and the data of Figure 4e for the waiting time.

In [1]:
import json

import numpy as np

from fixloc.make.si import quote

load = lambda f: json.load(open(f))
pct = '{:.0%}'
qs = ('0.25', '0.5', '0.75', '0.9', 'exp')
a7 = load('watch_all7_summary.json')
a9 = {r: load(f'monitor_N9_r{r}_summary.json') for r in ('1.25', '2', '10')}

## Graphs with $N=7$ vertices

In [2]:
s = a7['2.0']
quote(
  r'At $r=2$ and the expected fixation time, the optimal vertex to monitor is the likeliest fixation location on $91\%$ of the connected undirected graphs with $N=7$ vertices.',
  ('91%', s['share_rank1_exp'], pct),
)
quote(
  r'The optimal vertex is the likeliest or the second likeliest fixation location on $96\%$ of the graphs.',
  ('96%', s['share_rank_le2_exp'], pct),
)
print(s['graphs'], 'graphs, of which', s['skipped'], 'with all vertices equivalent are skipped')

At r=2 and the expected fixation time, the optimal vertex to monitor is the likeliest fixation location on 91% of the connected undirected graphs with N=7 vertices.
    SI        91%   computed        91%   (0.905882)   ok

The optimal vertex is the likeliest or the second likeliest fixation location on 96% of the graphs.
    SI        96%   computed        96%   (0.962353)   ok

853 graphs, of which 3 with all vertices equivalent are skipped


In [3]:
v7 = {k: np.array([[a7[r][f'{k}_{q}'] for q in qs] for r in a7]) for k in ('share_rank1', 'share_rank_le2')}
dev = max(np.abs(v - a7['2.0'][f'{k}_exp']).max() for k, v in v7.items())
quote(
  r'Across the five sampling times and the three values $r=1.25$, $2$, and $10$, the shares change by at most four percentage points.',
  ('<= 4', 100 * dev, lambda x: '<= 4' if x <= 4 + 1e-9 else f'{x:.2f}'),
)
print(f'largest change from the shares at r = 2 and the expected fixation time: {100 * dev:.2f} points')
print(f'largest difference between two of the 15 values of one share: {100 * max(v.max() - v.min() for v in v7.values()):.2f} points')
for k, v in v7.items():
  print(k, 'in % (rows r = 1.25, 2, 10; columns f_t = 0.25, 0.5, 0.75, 0.9, expected time):', np.round(100 * v, 1).tolist())

Across the five sampling times and the three values r=1.25, 2, and 10, the shares change by at most four percentage points.
    SI       <= 4   computed       <= 4   (4)   ok

largest change from the shares at r = 2 and the expected fixation time: 4.00 points
largest difference between two of the 15 values of one share: 5.29 points
share_rank1 in % (rows r = 1.25, 2, 10; columns f_t = 0.25, 0.5, 0.75, 0.9, expected time): [[94.6, 92.9, 91.6, 90.9, 92.4], [92.6, 91.2, 89.9, 89.3, 90.6], [91.3, 91.3, 91.2, 90.6, 91.3]]
share_rank_le2 in % (rows r = 1.25, 2, 10; columns f_t = 0.25, 0.5, 0.75, 0.9, expected time): [[97.8, 97.3, 96.7, 96.5, 96.9], [96.8, 96.5, 95.9, 96.0, 96.2], [96.7, 97.1, 97.2, 97.1, 97.3]]


## Waiting time

In [4]:
e = load('../../fig4/e/data.json')
quote(r'Hence $f_t\geq0.95$ for $t\geq20T$.', ('0.95', 1 - 1 / 20, '{:.2f}'))
quote(
  r'For the true structure and the theorized structure of Figure~4\textbf{e} of the main text, the waiting time is $20\times91.64\approx1{,}833$ time steps.',
  ('91.64', max(e['T']), '{:.2f}'),
  ('1,833', 20 * max(e['T']), '{:,.0f}'),
)
quote(
  r'On the two structures of Figure~4\textbf{e} of the main text the fixation times at $r=2$ are about $91$ time steps.',
  ('91', e['T'][0], '{:.0f}'),
  ('91', e['T'][1], '{:.0f}'),
)
quote(r'The bound $rN^5/(r-1)$ for $N=7$ and $r=2$ is $33{,}614$ time steps.', ('33,614', 2 * 7**5 / (2 - 1), '{:,.0f}'))

Hence f_t>=0.95 for t>=20T.
    SI       0.95   computed       0.95   (0.95)   ok

For the true structure and the theorized structure of Figure 4e of the main text, the waiting time is 20 x 91.64 about 1,833 time steps.
    SI      91.64   computed      91.64   (91.6363)   ok
    SI      1,833   computed      1,833   (1832.73)   ok

On the two structures of Figure 4e of the main text the fixation times at r=2 are about 91 time steps.
    SI         91   computed         92   (91.6363)   DIFFERS
    SI         91   computed         91   (91.111)   ok

The bound rN^5/(r-1) for N=7 and r=2 is 33,614 time steps.
    SI     33,614   computed     33,614   (33614)   ok



## The source and sink structure of Figure 1

In [5]:
from fixloc.make import fig1

pairs = lambda sep: lambda E: ', '.join(f'{a}{sep}{b}' for a, b in E)
quote(
  r'The directed edges are $0\to3$, $0\to4$, $1\to5$, $1\to6$, $2\to6$, $2\to7$, and $12\to2$.',
  ('0>3, 0>4, 1>5, 1>6, 2>6, 2>7, 12>2', sorted(fig1.ONE), pairs('>')),
)
quote(
  r'The rungs of the ladder are $3$--$8$, $4$--$9$, $6$--$11$, and $7$--$12$.',
  ('3-8, 4-9, 6-11, 7-12', sorted(e for e in fig1.TWO if abs(e[0] - e[1]) == 5), pairs('-')),
)

The directed edges are 0 -> 3, 0 -> 4, 1 -> 5, 1 -> 6, 2 -> 6, 2 -> 7, and 12 -> 2.
    SI 0>3, 0>4, 1>5, 1>6, 2>6, 2>7, 12>2   computed 0>3, 0>4, 1>5, 1>6, 2>6, 2>7, 12>2   ([(0, 3), (0, 4), (1, 5), (1, 6), (2, 6), (2, 7), (12, 2)])   ok

The rungs of the ladder are 3-8, 4-9, 6-11, and 7-12.
    SI 3-8, 4-9, 6-11, 7-12   computed 3-8, 4-9, 6-11, 7-12   ([(3, 8), (4, 9), (6, 11), (7, 12)])   ok



In [6]:
f1 = load('fig1_sources.json')
th, rho = np.array(f1['theta']), np.array(f1['rho'])
src = np.array(f1['sources'])
sink = np.setdiff1d(np.arange(len(th)), src)
quote(r'The structure has $N=13$ vertices.', ('13', len(th), '{}'))
quote(r'The source vertices $0$ and $1$ have temperature $1/2$.', ('0.5', th[0], '{:g}'), ('0.5', th[1], '{:g}'))
quote(
  r'The sink vertices have temperatures between $2/3$ and $11/6$.',
  ('0.6667', th[sink].min(), '{:.4f}'),
  ('1.8333', th[sink].max(), '{:.4f}'),
)
quote(
  r'At $r=1$ the fixation probability is between $0.149$ and $0.196$ from a source vertex and between $0.015$ and $0.115$ from a sink vertex.',
  ('0.149', rho[src].min(), '{:.3f}'),
  ('0.196', rho[src].max(), '{:.3f}'),
  ('0.015', rho[sink].min(), '{:.3f}'),
  ('0.115', rho[sink].max(), '{:.3f}'),
)
quote(
  r'Conditional on fixation at $r=1$, the initial vertex is a source vertex with probability $0.535$.',
  ('0.535', rho[src].sum() / rho.sum(), '{:.3f}'),
)
quote(
  r'The fixation location distribution and the monitoring probabilities are exact, from the chain on all $2^{13}$ sets of mutants.',
  ('2^13', len(th), lambda n: f'2^{n}'),
)
print('temperatures', np.round(th, 4).tolist())

The structure has N=13 vertices.
    SI         13   computed         13   (13)   ok

The source vertices 0 and 1 have temperature 1/2.
    SI        0.5   computed        0.5   (0.5)   ok
    SI        0.5   computed        0.5   (0.5)   ok

The sink vertices have temperatures between 2/3 and 11/6.
    SI     0.6667   computed     0.6667   (0.666667)   ok
    SI     1.8333   computed     1.8333   (1.83333)   ok

At r=1 the fixation probability is between 0.149 and 0.196 from a source vertex and between 0.015 and 0.115 from a sink vertex.
    SI      0.149   computed      0.149   (0.148606)   ok
    SI      0.196   computed      0.196   (0.196479)   ok
    SI      0.015   computed      0.015   (0.0151206)   ok
    SI      0.115   computed      0.115   (0.115075)   ok

Conditional on fixation at r=1, the initial vertex is a source vertex with probability 0.535.
    SI      0.535   computed      0.535   (0.535233)   ok

The fixation location distribution and the monitoring probabilities 

## Graphs with $N=9$ vertices

In [7]:
quote(
  r'The $7$ graphs with all vertices equivalent are removed, leaving $261{,}073$ graphs.',
  ('7', a9['2']['all_equivalent_skipped'], '{}'),
  ('261,073', a9['2']['evaluated'], '{:,}'),
)
quote(
  r'At the expected fixation time the shares are $90\%$ and $97\%$ at $r=1.25$, $89\%$ and $96\%$ at $r=2$, and $93\%$ and $98\%$ at $r=10$.',
  ('90%', a9['1.25']['share_rank1_exp'], pct),
  ('97%', a9['1.25']['share_rank_le2_exp'], pct),
  ('89%', a9['2']['share_rank1_exp'], pct),
  ('96%', a9['2']['share_rank_le2_exp'], pct),
  ('93%', a9['10']['share_rank1_exp'], pct),
  ('98%', a9['10']['share_rank_le2_exp'], pct),
)
v9 = {k: np.array([[a9[r][f'{k}_{q}'] for q in qs] for r in a9]) for k in ('share_rank1', 'share_rank_le2')}
dev = max(np.abs(v - v[:, -1:]).max() for v in v9.values())
quote(
  r'Across the five sampling times the shares change by at most three percentage points.',
  ('<= 3', 100 * dev, lambda x: '<= 3' if x <= 3 + 1e-9 else f'{x:.2f}'),
)
print(f'largest change from the share at the expected fixation time, at each r: {100 * dev:.2f} points')
print(f'largest difference between two of the five values of one share at one r: {100 * max((v.max(1) - v.min(1)).max() for v in v9.values()):.2f} points')

The 7 graphs with all vertices equivalent are removed, leaving 261,073 graphs.
    SI          7   computed          7   (7)   ok
    SI    261,073   computed    261,073   (261073)   ok

At the expected fixation time the shares are 90% and 97% at r=1.25, 89% and 96% at r=2, and 93% and 98% at r=10.
    SI        90%   computed        90%   (0.897634)   ok
    SI        97%   computed        97%   (0.968737)   ok
    SI        89%   computed        89%   (0.887759)   ok
    SI        96%   computed        96%   (0.964906)   ok
    SI        93%   computed        93%   (0.930541)   ok
    SI        98%   computed        98%   (0.9841)   ok

Across the five sampling times the shares change by at most three percentage points.
    SI       <= 3   computed       <= 3   (2.21815)   ok

largest change from the share at the expected fixation time, at each r: 2.22 points
largest difference between two of the five values of one share at one r: 3.52 points


## Simulated networks with $N=50$ to $200$ vertices

In [8]:
lw = load('large_watch_rank.json')
fam = {str(x['id']): x['family'] for x in load('../inference/large_summary.json')['rows']}
keys = [k.split(':') for k in lw]
quote(
  r'The simulated networks with $N=50$, $100$, and $200$ vertices are described in the section on inferring the initial mutant location.',
)
print('numbers of vertices', sorted({v['N'] for v in lw.values()}), '(the grids have 49, 100, and 196 vertices)')
quote(
  r'For the simulated networks the probabilities come from simulation at $r=1$, $1.5$, $2$, and $5$.',
  ('1, 1.5, 2, 5', sorted({float(r) for _, r in keys}), lambda x: ', '.join(f'{r:g}' for r in x)),
)
quote(
  r'From each initial vertex we simulated $500$ fixation trajectories per network and fitness and stored the mutant set at three times.',
  ('500', sorted({v['ck_takeovers'] // v['N'] for v in lw.values()}), lambda x: ', '.join(str(n) for n in x)),
)
quote(
  r'The three times are the times of fixation probability about $0.25$, $0.5$, and $0.75$, estimated from a pilot run.',
  *[(si, np.median([v[f'ck{k}']['f'] for v in lw.values()]), '{:.2f}') for k, si in enumerate(('0.25', '0.50', '0.75'))],
)
quote(
  r'The fixation location distribution comes from $4{,}000$ further fixation trajectories per initial vertex.',
  ('4,000', sorted({v['takeovers'] // v['N'] for v in lw.values()}), lambda x: ', '.join(f'{n:,}' for n in x)),
)

The simulated networks with N=50, 100, and 200 vertices are described in the section on inferring the initial mutant location.

numbers of vertices [49, 50, 100, 196, 200] (the grids have 49, 100, and 196 vertices)
For the simulated networks the probabilities come from simulation at r=1, 1.5, 2, and 5.
    SI 1, 1.5, 2, 5   computed 1, 1.5, 2, 5   ([1.0, 1.5, 2.0, 5.0])   ok

From each initial vertex we simulated 500 fixation trajectories per network and fitness and stored the mutant set at three times.
    SI        500   computed        500   ([500])   ok

The three times are the times of fixation probability about 0.25, 0.5, and 0.75, estimated from a pilot run.
    SI       0.25   computed       0.25   (0.250021)   ok
    SI       0.50   computed       0.50   (0.499508)   ok
    SI       0.75   computed       0.75   (0.750024)   ok

The fixation location distribution comes from 4,000 further fixation trajectories per initial vertex.
    SI      4,000   computed      4,000   ([4000]

In [9]:
rows = [(n, float(r), lw[f'{n}:{r}']) for n, r in keys if n != '60' and fam[n] not in ('cycle', 'torus')]
c = [v['ck1'] for _, _, v in rows]
rank = np.array([x['rank'] for x in c])
f, Fb, Fl = (np.array([x[k] for x in c]) for k in ('f', 'F_best', 'F_likeliest'))
by_r = {r: np.array([v['ck1']['rank'] for _, rr, v in rows if rr == r]) for r in (1.0, 1.5, 2.0, 5.0)}
by_f = {g: np.array([v['ck1']['rank'] for n, _, v in rows if fam[n] == g]) for g in sorted(set(fam.values()))}
quote(r'On the remaining $180$ network and fitness cases we take the time of fixation probability $0.5$.', ('180', len(rows), '{}'))
quote(
  r'The vertex with the largest $f_t/m_w(t)$ is the likeliest fixation location on $70\%$ of the cases.',
  ('70%', np.mean(rank == 1), pct),
)
quote(
  r'The vertex with the largest $f_t/m_w(t)$ is the likeliest or second likeliest fixation location on $82\%$ of the cases.',
  ('82%', np.mean(rank <= 2), pct),
)
quote(
  r'The lowest shares are under neutral evolution ($44\%$) and on random trees ($50\%$).',
  ('44%', np.mean(by_r[1.0] == 1), pct),
  ('50%', np.mean(by_f['random tree'] == 1), pct),
)
quote(r'The highest share is at $r=5$ ($83\%$).', ('83%', np.mean(by_r[5.0] == 1), pct))
print('by r:', {r: f'{np.mean(v == 1):.0%} of {len(v)}' for r, v in by_r.items()})
print('by family:', {g: f'{np.mean(v == 1):.0%} of {len(v)}' for g, v in by_f.items() if len(v)})

On the remaining 180 network and fitness cases we take the time of fixation probability 0.5.
    SI        180   computed        180   (180)   ok

The vertex with the largest f_t/m_w(t) is the likeliest fixation location on 70% of the cases.
    SI        70%   computed        70%   (0.7)   ok

The vertex with the largest f_t/m_w(t) is the likeliest or second likeliest fixation location on 82% of the cases.
    SI        82%   computed        82%   (0.822222)   ok

The lowest shares are under neutral evolution (44%) and on random trees (50%).
    SI        44%   computed        44%   (0.444444)   ok
    SI        50%   computed        50%   (0.5)   ok

The highest share is at r=5 (83%).
    SI        83%   computed        83%   (0.833333)   ok

by r: {1.0: '44% of 18', 1.5: '63% of 54', 2.0: '72% of 54', 5.0: '83% of 54'}
by family: {'Barabasi-Albert': '70% of 30', 'Erdos-Renyi': '93% of 30', 'random 3-regular': '70% of 30', 'random directed': '70% of 30', 'random tree': '50% of 30', '

In [10]:
frac = (Fl - f) / np.maximum(Fb - f, 1e-12)
quote(
  r'On $87\%$ of the cases the rise at the likeliest fixation location is at least nine tenths of the rise at the optimal vertex.',
  ('87%', np.mean(frac >= 0.9), pct),
)
quote(
  r'The difference between the two raised probabilities is at most $0.034$ on every case.',
  ('0.034', (Fb - Fl).max(), '{:.3f}'),
)
quote(r'The rise itself is small on large networks, $0.056$ in the median case.', ('0.056', np.median(Fb - f), '{:.3f}'))

On 87% of the cases the rise at the likeliest fixation location is at least nine tenths of the rise at the optimal vertex.
    SI        87%   computed        87%   (0.872222)   ok

The difference between the two raised probabilities is at most 0.034 on every case.
    SI      0.034   computed      0.034   (0.0339875)   ok

The rise itself is small on large networks, 0.056 in the median case.
    SI      0.056   computed      0.056   (0.0562081)   ok



## Two monitored vertices on graphs with $N=10$ vertices
At each of the times with $f_t$ at $0.25$, $0.5$, and $0.75$, the pair with the smallest probability of mutants at both
vertices. The likeliest fixation locations are the vertices whose fixation location probability is within $10^{-9}$
of the largest.

In [11]:
quote(r'Consider cycles, grids, and random regular graphs with $N=10$ vertices.')
quote(r'The pair of vertices least likely to be occupied by mutants at the same time consists of two distant vertices.')
quote(r'The pair is not the two likeliest fixation locations, because the remaining wild types are clustered.')
for x in load('pairs_N10.json'):
  D, P = np.array(x['distances']), np.array(x['P_last'])
  top = np.flatnonzero(P > P.max() - 1e-9)
  second = np.flatnonzero(P > np.sort(P)[-3] + 1e-9) if len(top) == 1 else top
  for q, v in x['least_mutant_pair'].items():
    a, b = v['pair']
    print(
      f"{x['family']:16s} r = {x['r']:g}  f_t = {q:4s}  pair {a}, {b}  distance {D[a, b]} of at most {D.max()}",
      f'  likeliest fixation locations {top.tolist()}  both vertices among the two likeliest: {a in second and b in second}',
    )

Consider cycles, grids, and random regular graphs with N=10 vertices.

The pair of vertices least likely to be occupied by mutants at the same time consists of two distant vertices.

The pair is not the two likeliest fixation locations, because the remaining wild types are clustered.

cycle            r = 1  f_t = 0.25  pair 0, 5  distance 5 of at most 5   likeliest fixation locations [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  both vertices among the two likeliest: True
cycle            r = 1  f_t = 0.5   pair 0, 5  distance 5 of at most 5   likeliest fixation locations [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  both vertices among the two likeliest: True
cycle            r = 1  f_t = 0.75  pair 0, 5  distance 5 of at most 5   likeliest fixation locations [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  both vertices among the two likeliest: True
cycle            r = 2  f_t = 0.25  pair 1, 6  distance 5 of at most 5   likeliest fixation locations [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  both vertices among the two likeliest: True
cy